# Part 6 — 공휴일/명절 효과 분석

공휴일 정보 API를 통해 2004~2024년 공휴일 데이터를 조회하고,
공휴일 전후 공실률 패턴을 분석한다.

## 0. 설정

In [8]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import numpy as np
import requests

# .env 위치를 상위 디렉터리까지 탐색하여 프로젝트 루트 결정
# → Jupyter Lab에서 실행해도, jupyter execute로 실행해도 동작
_dotenv_path = find_dotenv()
load_dotenv(_dotenv_path)
ROOT_DIR = Path(_dotenv_path).parent  # gyotong/

SERVICE_KEY = os.getenv("HOLIDAY_API_KEY")
if SERVICE_KEY is None:
    raise ValueError(
        ".env 파일에 HOLIDAY_API_KEY를 설정해주세요.\n"
        "예) HOLIDAY_API_KEY=abc123def456..."
    )

print(f"✅ API Key 로드 완료 (prefix: {SERVICE_KEY[:4]}...)")
print(f"📁 ROOT_DIR: {ROOT_DIR}")

✅ API Key 로드 완료 (prefix: f944...)
📁 ROOT_DIR: /home/sms/openclaw_file/korail_project/vacant/gyotong


## 1. 공휴일 데이터 API 조회

데이터 범위: 2004년 4월 ~ 2024년 12월 (ktx_long.csv 범위와 일치)

In [9]:
HOLIDAY_API_URL = "http://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService/getRestDeInfo"


def fetch_holidays_by_year(year: int) -> dict:
    """단일 연도 공휴일 조회"""
    params = {
        "serviceKey": SERVICE_KEY,
        "solYear": year,      # yearStr → solYear
        "_type": "json",      # 누락 시 XML 반환 → resp.json() 에러
        "numOfRows": 50,      # 기본 10개 → 연도당 최대 ~20건이므로 50으로 충분
    }
    resp = requests.get(HOLIDAY_API_URL, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()


def parse_holiday_response(data: dict) -> pd.DataFrame:
    """API 응답에서 공휴일 목록을 DataFrame으로 파싱합니다."""
    try:
        items = data.get("response", {}).get("body", {}).get("items", {}).get("item", [])
    except Exception:
        items = []

    # 단건 반환 시 dict로 오는 경우 처리 (공공데이터포털 공통 패턴)
    if isinstance(items, dict):
        items = [items]

    if not items:
        return pd.DataFrame(columns=["date", "name", "year", "month", "day"])

    rows = []
    for item in items:
        date_int = item.get("locdate")   # schulDate1 → locdate (정수형 ex. 20240101)
        name = item.get("dateName", "")  # schulName1 → dateName

        if not date_int:
            continue

        try:
            dt = datetime.strptime(str(date_int), "%Y%m%d")
            rows.append({
                "date": dt.date(),
                "name": name,
                "year": dt.year,
                "month": dt.month,
                "day": dt.day,
            })
        except ValueError:
            continue

    return pd.DataFrame(rows)

In [10]:
# 2004~2024년 공휴일 전체 조회
all_holidays = []
start_year = 2004
end_year = 2024

print(f"📡 {start_year}~{end_year}년 공휴일 조회 시작...")
for year in range(start_year, end_year + 1):
    try:
        raw = fetch_holidays_by_year(year)
        df = parse_holiday_response(raw)
        all_holidays.append(df)
        print(f"  {year}년: {len(df)}개 공휴일")
    except Exception as e:
        print(f"  {year}년: 조회 실패 - {e}")

holidays_df = pd.concat(all_holidays, ignore_index=True)
print(f"\n✅ 총 {len(holidays_df)}개 공휴일 조회 완료")
print(holidays_df.head(10))

📡 2004~2024년 공휴일 조회 시작...
  2004년: 16개 공휴일
  2005년: 16개 공휴일
  2006년: 15개 공휴일
  2007년: 15개 공휴일
  2008년: 14개 공휴일
  2009년: 14개 공휴일
  2010년: 14개 공휴일
  2011년: 14개 공휴일
  2012년: 16개 공휴일
  2013년: 15개 공휴일
  2014년: 17개 공휴일
  2015년: 16개 공휴일
  2016년: 18개 공휴일
  2017년: 19개 공휴일
  2018년: 18개 공휴일
  2019년: 16개 공휴일
  2020년: 18개 공휴일
  2021년: 18개 공휴일
  2022년: 19개 공휴일
  2023년: 18개 공휴일
  2024년: 19개 공휴일

✅ 총 345개 공휴일 조회 완료
         date   name  year  month  day
0  2004-01-01     신정  2004      1    1
1  2004-01-21     설날  2004      1   21
2  2004-01-22     설날  2004      1   22
3  2004-01-23     설날  2004      1   23
4  2004-03-01    삼일절  2004      3    1
5  2004-04-05    식목일  2004      4    5
6  2004-05-05   어린이날  2004      5    5
7  2004-05-26  석가탄신일  2004      5   26
8  2004-06-06    현충일  2004      6    6
9  2004-07-17    제헌절  2004      7   17


## 1-1. 캐시 저장 / 로드

API를 매번 호출하지 않도록 CSV로 캐시합니다.

In [ ]:
print(f"✅ {len(holidays_df)}개 공휴일 조회 완료")
print(holidays_df.to_string(index=False))

## 2. 명절 분류

공휴일을 명절 (설날·추석)과 일반 공휴일로 분류합니다.

In [12]:
# 명절 키워드 정의
# 한식·동지는 법정 공휴일이 아니므로 API에서 반환되지 않음
SEASON_KEYWORDS = ["설날", "구정", "추석"]

def classify_holiday(name: str) -> str:
    """명절 vs 일반 공휴일 분류"""
    if isinstance(name, float):
        return "일반"
    name = str(name)
    for kw in SEASON_KEYWORDS:
        if kw in name:
            return "명절"
    return "일반"


holidays_df["유형"] = holidays_df["name"].apply(classify_holiday)

print("공휴일 유형별 개수:")
print(holidays_df["유형"].value_counts())
print()
print("명절 목록:")
major = holidays_df[holidays_df["유형"] == "명절"]
print(major.to_string(index=False))

공휴일 유형별 개수:
유형
일반    218
명절    127
Name: count, dtype: int64

명절 목록:
      date      name  year  month  day 유형
2004-01-21        설날  2004      1   21 명절
2004-01-22        설날  2004      1   22 명절
2004-01-23        설날  2004      1   23 명절
2004-09-27        추석  2004      9   27 명절
2004-09-28        추석  2004      9   28 명절
2004-09-29        추석  2004      9   29 명절
2005-02-08        설날  2005      2    8 명절
2005-02-09        설날  2005      2    9 명절
2005-02-10        설날  2005      2   10 명절
2005-09-17        추석  2005      9   17 명절
2005-09-18        추석  2005      9   18 명절
2005-09-19        추석  2005      9   19 명절
2006-01-28        설날  2006      1   28 명절
2006-01-29        설날  2006      1   29 명절
2006-01-30        설날  2006      1   30 명절
2006-10-05        추석  2006     10    5 명절
2006-10-06        추석  2006     10    6 명절
2006-10-07        추석  2006     10    7 명절
2007-02-17        설날  2007      2   17 명절
2007-02-18        설날  2007      2   18 명절
2007-02-19        설날  2007      2   19 명절
2007-09

## 3. 공휴일 Feature 생성

ktx_long.csv (월별 데이터)에 공휴일 관련 컬럼을 추가합니다.

In [13]:
# KTX 데이터 로드
ktx = pd.read_csv(
    ROOT_DIR / "data/processed/ktx_long.csv",
    encoding="utf-8-sig",
    parse_dates=["yearmonth"],
)

print(f"KTX 데이터: {ktx.shape}")
print(f"연월 범위: {ktx['yearmonth'].min()} ~ {ktx['yearmonth'].max()}")

KTX 데이터: (954, 16)
연월 범위: 2004-04-01 00:00:00 ~ 2024-12-01 00:00:00


In [14]:
def add_holiday_features(df: pd.DataFrame, holidays: pd.DataFrame) -> pd.DataFrame:
    """
    월별 데이터에 공휴일 관련 feature를 추가합니다.
    
    생성되는 컬럼:
    - 공휴일_개수: 해당 월의 공휴일 수
    - 명절_여부: 해당 월에 명절이 있으면 1
    - 명절_유형: 설날 / 추석 / 한식 / 없음
    """
    result = df.copy()

    holiday_counts = []
    has_major = []
    major_type = []

    for _, row in result.iterrows():
        ym = row["yearmonth"]
        y, m = ym.year, ym.month

        # 해당 월의 공휴일 필터링
        month_holidays = holidays[(holidays["year"] == y) & (holidays["month"] == m)]

        holiday_counts.append(len(month_holidays))

        # 명절 여부
        major_holidays = month_holidays[month_holidays["유형"] == "명절"]
        has_major.append(1 if len(major_holidays) > 0 else 0)

        # 명절 유형
        names = major_holidays["name"].str.lower() if len(major_holidays) > 0 else pd.Series([])
        if "설" in " ".join(names) or "구정" in " ".join(names):
            major_type.append("설날")
        elif "추석" in " ".join(names):
            major_type.append("추석")
        elif "한식" in " ".join(names):
            major_type.append("한식")
        else:
            major_type.append("없음")

    result["공휴일_개수"] = holiday_counts
    result["명절_여부"] = has_major
    result["명절_유형"] = major_type
    return result


ktx_holiday = add_holiday_features(ktx, holidays_df)
print(f"✅ 공휴일 feature 추가 완료: {ktx_holiday.shape}")
print()
print("명절_유형별 분포:")
print(ktx_holiday["명절_유형"].value_counts())
print()
print("공휴일_개수 분포:")
print(ktx_holiday["공휴일_개수"].describe().round(1))

✅ 공휴일 feature 추가 완료: (954, 19)

명절_유형별 분포:
명절_유형
없음    779
추석     89
설날     86
Name: count, dtype: int64

공휴일_개수 분포:
count    954.0
mean       1.4
std        1.3
min        0.0
25%        0.0
50%        1.0
75%        2.0
max        7.0
Name: 공휴일_개수, dtype: float64


## 4. 공휴일 전후 공실률 패턴 분석

명절 전후 ±1개월 범위에서 공실률이 어떻게 변화하는지 시각화합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 한글 폰트 설정
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if Path(font_path).exists():
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams["font.family"] = prop.get_name()
else:
    print(f"⚠️ CJK 폰트를 {font_path}에서 찾지 못했습니다.")

In [ ]:
def plot_holiday_vacancy_pattern(df: pd.DataFrame):
    """
    명절 월 vs 비명절 월의 공실률 분포를 Boxplot으로 비교합니다.
    """
    df = df.copy()
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    df["비교그룹"] = df["명절_여부"].map({1: "명절월", 0: "일반월"})
    sns.boxplot(
        data=df,
        x="비교그룹",
        y="공실률",
        palette={"명절월": "#e74c3c", "일반월": "#3498db"},
        ax=axes[0],
    )
    axes[0].set_title("명절월 vs 일반월 공실률 분포")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("공실률 (%)")

    for i, group in enumerate(["명절월", "일반월"]):
        subset = df[df["비교그룹"] == group]["공실률"]
        axes[0].text(
            i, subset.max() + 2,
            f"avg={subset.mean():.1f}\nmed={subset.median():.1f}",
            ha="center", fontsize=10,
        )

    major_df = df[df["명절_유형"] != "없음"]
    if len(major_df) > 0:
        order = ["설날", "추석"]
        existing = [t for t in order if t in major_df["명절_유형"].values]
        sns.boxplot(
            data=major_df,
            x="명절_유형",
            y="공실률",
            order=existing,
            palette={"설날": "#e67e22", "추석": "#9b59b6"},
            ax=axes[1],
        )
        axes[1].set_title("명절 유형별 공실률 분포")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("공실률 (%)")
    else:
        axes[1].text(0.5, 0.5, "명절 데이터 없음", ha="center", va="center", fontsize=14)
        axes[1].set_title("명절 유형별 공실률")

    plt.tight_layout()
    plt.show()


plot_holiday_vacancy_pattern(ktx_holiday)

In [ ]:
def plot_route_holiday_effect(df: pd.DataFrame):
    """
    노선별 명절 효과 분석:
    각 노선에서 명절월 공실률 vs 일반월 공실률 차이
    """
    routes = sorted(df["노선"].unique())

    effect_data = []
    for route in routes:
        route_df = df[df["노선"] == route]
        holiday_avg = route_df[route_df["명절_여부"] == 1]["공실률"].mean()
        normal_avg = route_df[route_df["명절_여부"] == 0]["공실률"].mean()
        effect = holiday_avg - normal_avg
        effect_data.append({
            "노선": route,
            "명절월_평균": holiday_avg,
            "일반월_평균": normal_avg,
            "명절_효과": effect,
        })

    effect_df = pd.DataFrame(effect_data)
    print("=== 노선별 명절 효과 (명절월 - 일반월) ===")
    print(effect_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(routes))
    ax.bar([i - 0.2 for i in x], effect_df["명절월_평균"], 0.4,
           label="명절월 평균", color="#e74c3c", alpha=0.8)
    ax.bar([i + 0.2 for i in x], effect_df["일반월_평균"], 0.4,
           label="일반월 평균", color="#3498db", alpha=0.8)

    ax.set_xticks(list(x))
    ax.set_xticklabels(routes)
    ax.set_ylabel("공실률 (%)")
    ax.set_title("노선별 명절월 vs 일반월 공실률")
    ax.legend()

    for i, (_, row) in enumerate(effect_df.iterrows()):
        max_val = max(row["명절월_평균"], row["일반월_평균"])
        ax.text(i, max_val + 1, f"Δ{row['명절_효과']:+.1f}",
                ha="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_route_holiday_effect(ktx_holiday)

## 5. 공휴일 Feature → Model Input 통합

생성한 공휴일 feature를 model input CSV에 반영합니다.

In [ ]:
print(f"shape: {ktx_holiday.shape}")
print()
new_cols = ["공휴일_개수", "명절_여부", "명절_유형"]
print(f"추가된 컬럼: {new_cols}")
print()
print(ktx_holiday[["yearmonth", "노선"] + new_cols].head(15).to_string(index=False))

## 6. 요약

| 산출물 | 위치 |
|--------|------|
| 공휴일 캐시 | `data/processed/holidays_cache.csv` |
| 공휴일 포함 KTX 데이터 | `data/processed/ktx_long_with_holiday.csv` |
| 명절 vs 일반월 비교 차트 | `data/processed/eda_results/holiday_vacancy_comparison.png` |
| 노선별 명절 효과 차트 | `data/processed/eda_results/route_holiday_effect.png` |

### 다음 작업
- `src/features.py`의 `make_features()`에 공휴일 feature 통합
- `src/model.py`의 `FEATURE_COLS`에 `공휴일_개수`, `명절_여부` 추가
- 재학습 후 MAE 개선 여부 확인